# Activation Diffing

Compare activations between two inputs: paired layer-by-layer comparison, change localization, divergence mapping across all positions, causal change attribution, and minimal change identification.

## Why This Matters

Activation diffing characterizes the patterns and statistics of internal model activations. Activation structure reveals how the model processes information — which components are active, how sparse the computation is, and what geometric patterns emerge.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_diffing import (
    paired_activation_comparison,
    change_localization,
    divergence_mapping,
    causal_change_attribution,
    minimal_change_identification,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens_a = jnp.array([1, 15, 30, 45, 60])
tokens_b = jnp.array([50, 65, 80, 95, 10])
print('Model ready')

In [ ]:
# Paired comparison of activations at each layer
result = paired_activation_comparison(model, tokens_a, tokens_b)
print(f'Layers compared: {result["n_layers"]}')
for layer in result['per_layer']:
    print(f'  Layer {layer["layer"]}: L2={layer["l2_distance"]:.4f}, '
          f'cos={layer["cosine_similarity"]:.4f}, '
          f'max_dim={layer["max_dim_diff"]}')

In [ ]:
# Localize where changes are largest
result = change_localization(model, tokens_a, tokens_b)
print(f'Most changed component: {result["most_changed_component"]}')
print(f'Total attn change: {result["attn_total_change"]:.4f}')
print(f'Total MLP change: {result["mlp_total_change"]:.4f}')
for comp in result['component_changes'][:5]:
    print(f'  {comp["component"]}: {comp["diff"]:.4f}')

In [ ]:
# Map divergence across all layers and positions
result = divergence_mapping(model, tokens_a, tokens_b)
print(f'Divergence matrix shape: {result["divergence_matrix"].shape}')
print(f'Peak divergence: layer={result["peak_divergence"][0]}, '
      f'pos={result["peak_divergence"][1]}, '
      f'value={result["peak_divergence"][2]:.4f}')
print(f'Onset layer: {result["divergence_onset_layer"]}')
print(f'Mean per layer: {np.array(result["mean_per_layer"]).round(3)}')

In [ ]:
# Causal attribution of output changes to components
def metric_fn(logits, tokens):
    return jnp.mean(logits[-1])

result = causal_change_attribution(model, tokens_a, tokens_b, metric_fn)
print(f'Total change: {result["total_change"]:.4f}')
print('Top components:')
for comp in result['top_components'][:5]:
    print(f'  {comp["component"]}: attribution={comp["attribution"]:.4f}, '
          f'fraction={comp["fraction"]:.4f}')

In [ ]:
# Identify minimal dimensions explaining the change
result = minimal_change_identification(model, tokens_a, tokens_b, top_k=5)
print(f'Total diff norm: {result["total_diff_norm"]:.4f}')
print('Top dimensions:')
for dim, val in result['top_dimensions']:
    print(f'  Dim {dim}: diff={val:.4f}')
print(f'Cumulative explanation: {[f"{x:.3f}" for x in result["cumulative_explanation"]]}')